In [20]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
pd.set_option("display.max_columns", 300)
np.random.seed(42)

In [22]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "cleaned_project_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Project root not found (missing data/cleaned_project_dataset.csv)")

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "data"
RAW = DATA / "raw_unprocessed_data"

MATCH_PATH = DATA / "cleaned_project_dataset.csv"
SUMMARY_PATH = RAW / "FIFA - World Cup Summary.csv"

matches = pd.read_csv(MATCH_PATH)
summary = pd.read_csv(SUMMARY_PATH)

In [23]:
print("Matches:", matches.shape)
print("Summary:", summary.shape)

matches.head()

matches.dtypes

Matches: (966, 30)
Summary: (22, 9)


year                  int64
date                 object
tournament_id        object
tournament_name      object
match_name           object
stage                object
home_team            object
away_team            object
home_team_code       object
away_team_code       object
home_goals          float64
away_goals          float64
score                object
result               object
home_team_win       float64
away_team_win       float64
draw                 object
extra_time           object
penalty_shootout     object
score_penalties      object
stadium              object
stadium_id           object
stadium_name         object
city                 object
host_country         object
host_team            object
attendance          float64
match_time           object
referee              object
notes                object
dtype: object

# Data Cleaning

Purpose: Normalize types and column names, and filter to pre-2026 matches.

In [24]:
# Define cleaning helpers and apply them to matches and summary tables.
def clean_matches(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df = df[df["year"] < 2026].copy()

    df["stage"] = df["stage"].astype(str).str.strip()
    df["tournament_name"] = df["tournament_name"].astype(str).str.strip()

    return df


def clean_summary(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    df = df.rename(
        columns={
            "year": "year",
            "host": "host_country_summary",
            "champion": "champion",
            "runner_up": "runner_up",
            "third_place": "third_place",
            "teams": "total_teams",
            "matches_played": "matches_played",
            "goals_scored": "goals_scored_tournament",
            "avg_goals_per_game": "avg_goals_per_game",
        }
    )

    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

    return df


matches = clean_matches(matches)
summary = clean_summary(summary)

matches[["year", "date", "tournament_name", "stage"]].head()

,year,date,tournament_name,stage
0,1930,1930-07-13,1930 FIFA World Cup,Group stage
1,1930,1930-07-13,1930 FIFA World Cup,Group stage
2,1930,1930-07-14,1930 FIFA World Cup,Group stage
3,1930,1930-07-14,1930 FIFA World Cup,Group stage
4,1930,1930-07-15,1930 FIFA World Cup,Group stage


In [25]:
matches.dtypes

year                         int64
date                datetime64[ns]
tournament_id               object
tournament_name             object
match_name                  object
stage                       object
home_team                   object
away_team                   object
home_team_code              object
away_team_code              object
home_goals                 float64
away_goals                 float64
score                       object
result                      object
home_team_win              float64
away_team_win              float64
draw                        object
extra_time                  object
penalty_shootout            object
score_penalties             object
stadium                     object
stadium_id                  object
stadium_name                object
city                        object
host_country                object
host_team                   object
attendance                 float64
match_time                  object
referee             

In [26]:
# Create a stable match_id key used across all derived tables.
matches = matches.reset_index(drop=True).copy()
matches["match_id"] = np.arange(len(matches))

In [27]:
matches

,year,date,tournament_id,tournament_name,match_name,stage,home_team,away_team,home_team_code,away_team_code,home_goals,away_goals,score,result,home_team_win,away_team_win,draw,extra_time,penalty_shootout,score_penalties,stadium,stadium_id,stadium_name,city,host_country,host_team,attendance,match_time,referee,notes,match_id
0,1930,1930-07-13,WC-1930,1930 FIFA World Cup,France v Mexico,Group stage,France,Mexico,FRA,MEX,4.0,1.0,4-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Pocitos,S-193,Estadio Pocitos,Montevideo,Uruguay,0.0,4444.0,15:00,Domingo Lombardi,NaN,0
1,1930,1930-07-13,WC-1930,1930 FIFA World Cup,United States v Belgium,Group stage,United States,Belgium,USA,BEL,3.0,0.0,3-0,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,18346.0,15:00,Jose Macias,NaN,1
2,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Romania v Peru,Group stage,Romania,Peru,ROU,PER,3.0,1.0,3-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Pocitos,S-193,Estadio Pocitos,Montevideo,Uruguay,0.0,2549.0,14:50,Alberto Warnken,NaN,2
3,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Yugoslavia v Brazil,Group stage,Yugoslavia,Brazil,YUG,BRA,2.0,1.0,2-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,24059.0,12:45,Anibal Tejada,NaN,3
4,1930,1930-07-15,WC-1930,1930 FIFA World Cup,Argentina v France,Group stage,Argentina,France,ARG,FRA,1.0,0.0,1-0,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,23409.0,16:00,Gilberto Rego,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,2022,2022-12-10,WC-2022,2022 FIFA World Cup,NaN,quarter-finals,Morocco,Portugal,MAR,PRT,1.0,0.0,1-0,home team win,1.0,0.0,no,no,no,0-0,Al Thumama Stadium,S-112,NaN,Doha,Qatar,NaN,44198.0,18:00,Facundo Tello,NaN,857
858,2022,2022-12-13,WC-2022,2022 FIFA World Cup,NaN,semi-finals,Argentina,Croatia,ARG,HRV,3.0,0.0,3-0,home team win,1.0,0.0,no,no,no,0-0,Lusail Stadium,S-114,NaN,Lusail,Qatar,NaN,88966.0,22:00,Daniele Orsato,NaN,858
859,2022,2022-12-14,WC-2022,2022 FIFA World Cup,NaN,semi-finals,France,Morocco,FRA,MAR,2.0,0.0,2-0,home team win,1.0,0.0,no,no,no,0-0,Al Bayt Stadium,S-107,NaN,Al Khor,Qatar,NaN,68294.0,22:00,César Arturo Ramos,NaN,859
860,2022,2022-12-17,WC-2022,2022 FIFA World Cup,NaN,third-place match,Croatia,Morocco,HRV,MAR,2.0,1.0,2-1,home team win,1.0,0.0,no,no,no,0-0,Khalifa International Stadium,S-110,NaN,Al Rayyan,Qatar,NaN,44137.0,18:00,Abdulrahman Ibrahim Al Jassim,NaN,860


# Feature Engineering

Purpose: Build leakage-free team, season, and tournament features.

In [28]:
# Convert match-level data into a team-level long table for rolling metrics.
def build_long_table(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    df["match_id"] = np.arange(len(df))

    home = df[[
        "match_id",
        "date",
        "year",
        "stage",
        "tournament_name",
        "host_country",
        "attendance",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
    ]].copy()
    home = home.rename(
        columns={
            "home_team": "team",
            "away_team": "opponent",
            "home_goals": "goals_for",
            "away_goals": "goals_against",
        }
    )
    home["is_home"] = 1

    away = df[[
        "match_id",
        "date",
        "year",
        "stage",
        "tournament_name",
        "host_country",
        "attendance",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
    ]].copy()
    away = away.rename(
        columns={
            "away_team": "team",
            "home_team": "opponent",
            "away_goals": "goals_for",
            "home_goals": "goals_against",
        }
    )
    away["is_home"] = 0

    long_df = pd.concat([home, away], ignore_index=True)
    long_df = long_df.sort_values(["date", "match_id"]).reset_index(drop=True)

    valid_scores = long_df["goals_for"].notna() & long_df["goals_against"].notna()
    long_df["win"] = np.where(
        valid_scores & (long_df["goals_for"] > long_df["goals_against"]), 1, 0
    )
    long_df["draw"] = np.where(
        valid_scores & (long_df["goals_for"] == long_df["goals_against"]), 1, 0
    )
    long_df["loss"] = np.where(
        valid_scores & (long_df["goals_for"] < long_df["goals_against"]), 1, 0
    )

    long_df["points"] = long_df["win"] * 3 + long_df["draw"]

    stage_lower = long_df["stage"].astype(str).str.lower()
    long_df["is_group_stage"] = stage_lower.str.contains("group")
    long_df["is_knockout"] = ~long_df["is_group_stage"]

    return long_df


long_df = build_long_table(matches)
long_df.head()

,match_id,date,year,stage,tournament_name,host_country,attendance,team,opponent,goals_for,goals_against,is_home,win,draw,loss,points,is_group_stage,is_knockout
0,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,France,Mexico,4.0,1.0,1,1,0,0,3,True,False
1,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,Mexico,France,1.0,4.0,0,0,0,1,0,True,False
2,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,United States,Belgium,3.0,0.0,1,1,0,0,3,True,False
3,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,Belgium,United States,0.0,3.0,0,0,0,1,0,True,False
4,2,1930-07-14,1930,Group stage,1930 FIFA World Cup,Uruguay,2549.0,Romania,Peru,3.0,1.0,1,1,0,0,3,True,False


In [29]:
# Compute expanding and rolling team metrics used for pre-match features.
def compute_team_historical_metrics(long_df: pd.DataFrame) -> pd.DataFrame:
    df = long_df.copy()

    group = df.groupby("team", group_keys=False)

    df["total_matches_before"] = group.cumcount()
    df["total_wins_before"] = group["win"].cumsum().shift(1).fillna(0)
    df["total_draws_before"] = group["draw"].cumsum().shift(1).fillna(0)
    df["total_losses_before"] = group["loss"].cumsum().shift(1).fillna(0)

    df["total_goals_for_before"] = group["goals_for"].cumsum().shift(1).fillna(0)
    df["total_goals_against_before"] = group["goals_against"].cumsum().shift(1).fillna(0)

    df["goal_diff_before"] = df["total_goals_for_before"] - df["total_goals_against_before"]

    df["total_win_rate_before"] = df["total_wins_before"] / df["total_matches_before"].replace(0, np.nan)

    df["goal_diff_per_match_before"] = df["goal_diff_before"] / df["total_matches_before"].replace(0, np.nan)
    df["goals_per_match_before"] = df["total_goals_for_before"] / df["total_matches_before"].replace(0, np.nan)
    df["conceded_per_match_before"] = df["total_goals_against_before"] / df["total_matches_before"].replace(0, np.nan)

    # Recent form (last 5)
    df["last5_points"] = group["points"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
    df["last5_goal_diff"] = group.apply(
        lambda g: (g["goals_for"] - g["goals_against"]).shift(1).rolling(5, min_periods=1).sum()
    ).reset_index(level=0, drop=True)
    df["last5_win_rate"] = group["win"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

    # Stage-based win rates
    df["group_matches_before"] = group["is_group_stage"].cumsum().shift(1).fillna(0)
    df["group_wins_before"] = group.apply(
        lambda g: (g["win"] * g["is_group_stage"]).cumsum().shift(1).fillna(0)
    ).reset_index(level=0, drop=True)
    df["group_stage_win_rate_before"] = df["group_wins_before"] / df["group_matches_before"].replace(0, np.nan)

    df["knockout_matches_before"] = group["is_knockout"].cumsum().shift(1).fillna(0)
    df["knockout_wins_before"] = group.apply(
        lambda g: (g["win"] * g["is_knockout"]).cumsum().shift(1).fillna(0)
    ).reset_index(level=0, drop=True)
    df["knockout_win_rate_before"] = df["knockout_wins_before"] / df["knockout_matches_before"].replace(0, np.nan)

    # Home/Away performance
    df["times_played_as_home_before"] = group.apply(
        lambda g: g["is_home"].cumsum().shift(1).fillna(0)
    ).reset_index(level=0, drop=True)
    df["times_played_as_away_before"] = group.apply(
        lambda g: (1 - g["is_home"]).cumsum().shift(1).fillna(0)
    ).reset_index(level=0, drop=True)

    df["home_wins_before"] = group.apply(
        lambda g: (g["win"] * g["is_home"]).cumsum().shift(1).fillna(0)
    ).reset_index(level=0, drop=True)
    df["away_wins_before"] = group.apply(
        lambda g: (g["win"] * (1 - g["is_home"])).cumsum().shift(1).fillna(0)
    ).reset_index(level=0, drop=True)

    df["home_win_rate_before"] = df["home_wins_before"] / df["times_played_as_home_before"].replace(0, np.nan)
    df["away_win_rate_before"] = df["away_wins_before"] / df["times_played_as_away_before"].replace(0, np.nan)
    df["home_away_win_rate_diff"] = df["home_win_rate_before"] - df["away_win_rate_before"]

    # Attendance impact (historical average attendance)
    df["avg_attendance_before"] = group["attendance"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )

    return df


def compute_season_metrics(long_df: pd.DataFrame) -> pd.DataFrame:
    df = long_df.copy()
    group = df.groupby(["team", "year"], group_keys=False)

    df["matches_season_before"] = group.cumcount()
    df["wins_season_before"] = group["win"].cumsum().shift(1).fillna(0)
    df["goals_scored_season_before"] = group["goals_for"].cumsum().shift(1).fillna(0)
    df["goals_conceded_season_before"] = group["goals_against"].cumsum().shift(1).fillna(0)

    df["win_rate_season_before"] = df["wins_season_before"] / df["matches_season_before"].replace(0, np.nan)
    df["avg_goals_scored_season_before"] = df["goals_scored_season_before"] / df["matches_season_before"].replace(0, np.nan)
    df["avg_goals_conceded_season_before"] = df["goals_conceded_season_before"] / df["matches_season_before"].replace(0, np.nan)
    df["goal_diff_season_before"] = df["goals_scored_season_before"] - df["goals_conceded_season_before"]

    return df


long_hist = compute_team_historical_metrics(long_df)
long_hist = compute_season_metrics(long_hist)

long_hist[["team", "total_matches_before", "total_win_rate_before", "last5_win_rate", "matches_season_before"]]

/var/folders/kk/brw_pvj17nx5_4h1y3t17bkc0000gn/T/ipykernel_26052/1637921116.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["last5_goal_diff"] = group.apply(
/var/folders/kk/brw_pvj17nx5_4h1y3t17bkc0000gn/T/ipykernel_26052/1637921116.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["group_wins_before"] = group.apply(
/var/folders/kk/brw_pvj17nx5_4h1y3t17bkc0000gn/T/ipykernel_26052/1637921116.py:38: Fut

,team,total_matches_before,total_win_rate_before,last5_win_rate,matches_season_before
0,France,0,NaN,NaN,0
1,Mexico,0,NaN,NaN,0
2,United States,0,NaN,NaN,0
3,Belgium,0,NaN,NaN,0
4,Romania,0,NaN,NaN,0
...,...,...,...,...,...
1719,Morocco,19,1.894737,0.6,5
1720,Croatia,29,0.172414,0.2,6
1721,Morocco,20,0.650000,0.6,6
1722,Argentina,80,0.062500,0.8,6


In [30]:
long_hist

,match_id,date,year,stage,tournament_name,host_country,attendance,team,opponent,goals_for,goals_against,is_home,win,draw,loss,points,is_group_stage,is_knockout,total_matches_before,total_wins_before,total_draws_before,total_losses_before,total_goals_for_before,total_goals_against_before,goal_diff_before,total_win_rate_before,goal_diff_per_match_before,goals_per_match_before,conceded_per_match_before,last5_points,last5_goal_diff,last5_win_rate,group_matches_before,group_wins_before,group_stage_win_rate_before,knockout_matches_before,knockout_wins_before,knockout_win_rate_before,times_played_as_home_before,times_played_as_away_before,home_wins_before,away_wins_before,home_win_rate_before,away_win_rate_before,home_away_win_rate_diff,avg_attendance_before,matches_season_before,wins_season_before,goals_scored_season_before,goals_conceded_season_before,win_rate_season_before,avg_goals_scored_season_before,avg_goals_conceded_season_before,goal_diff_season_before
0,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,France,Mexico,4.0,1.0,1,1,0,0,3,True,False,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0.0
1,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,Mexico,France,1.0,4.0,0,0,0,1,0,True,False,0,1.0,0.0,0.0,4.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,-2.0,NaN,1.0,0.0,0.000000,0.0,0.0,NaN,1.0,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,0,1.0,4.0,1.0,NaN,NaN,NaN,3.0
2,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,United States,Belgium,3.0,0.0,1,1,0,0,3,True,False,0,0.0,0.0,1.0,1.0,4.0,-3.0,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,1.0,1.0,1.000000,0.0,0.0,NaN,2.0,0.0,1.0,0.0,0.500000,NaN,NaN,NaN,0,0.0,1.0,4.0,NaN,NaN,NaN,-3.0
3,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,Belgium,United States,0.0,3.0,0,0,0,1,0,True,False,0,1.0,0.0,0.0,3.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,1.0,1.0,1.000000,0.0,0.0,NaN,3.0,0.0,1.0,0.0,0.333333,NaN,NaN,NaN,0,1.0,3.0,0.0,NaN,NaN,NaN,3.0
4,2,1930-07-14,1930,Group stage,1930 FIFA World Cup,Uruguay,2549.0,Romania,Peru,3.0,1.0,1,1,0,0,3,True,False,0,0.0,0.0,1.0,0.0,3.0,-3.0,NaN,NaN,NaN,NaN,NaN,-2.0,NaN,1.0,1.0,1.000000,0.0,0.0,NaN,3.0,1.0,1.0,0.0,0.333333,0.000000,0.333333,NaN,0,0.0,0.0,3.0,NaN,NaN,NaN,-3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1719,859,2022-12-14,2022,semi-finals,2022 FIFA World Cup,Qatar,68294.0,Morocco,France,0.0,2.0,0,0,0,1,0,False,True,19,36.0,11.0,17.0,115.0,67.0,48.0,1.894737,2.526316,6.052632,3.526316,11.0,4.0,0.6,42.0,9.0,0.214286,22.0,2.0,0.090909,12.0,13.0,10.0,1.0,0.833333,0.076923,0.756410,41300.055556,5,5.0,13.0,5.0,1.000000,2.600000,1.000000,8.0
1720,860,2022-12-17,2022,third-place match,2022 FIFA World Cup,Qatar,44137.0,Croatia,Morocco,2.0,1.0,1,1,0,0,3,False,True,29,5.0,7.0,8.0,18.0,22.0,-4.0,0.172414,-0.137931,0.620690,0.758621,6.0,5.0,0.2,17.0,9.0,0.529412,3.0,3.0,1.000000,12.0,14.0,10.0,2.0,0.833333,0.142857,0.690476,49296.689655,6,3.0,5.0,3.0,0.500000,0.833333,0.500000,2.0
1721,860,2022-12-17,2022,third-place match,2022 FIFA World Cup,Qatar,44137.0,Morocco,Croatia,1.0,2.0,0,0,0,1,0,False,True,20,13.0,8.0,9.0,43.0,33.0,10.0,0.650000,0.500000,2.150000,1.650000,10.0,7.0,0.6,18.0,10.0,0.555556,12.0,3.0,0.250000,13.0,14.0,11.0,2.0,0.846154,0.142857,0.703297,42720.789474,6,2.0,8.0,7.0,0.333333,1.333333,1.166667,1.0
1722,861,2022-12-18,2022,final,2022 FIFA World Cup,Qatar,88966.0,Argentina,France,3.0,3.0,1,0,1,0,1,False,True,80,5.0,7.0,9.0,19.0,24.0,-5.0,0.062500,-0.062500,0.237500,0.300000,13.0,6.0,0.8,17.0,10.0,0.588235,4.0,3.0,0.750000,13.0,15.0,11.0,2.0,0.846154,0.133333,0.712821,53043.763158,6,3.0,6.0,5.0,0.500000,1.000000,0.833333,1.0


In [32]:
# Add tournament context, champion history, H2H, and Elo features.
def compute_tournament_experience(long_df: pd.DataFrame) -> pd.DataFrame:
    df = long_df.copy()
    team_year = df[["team", "year"]].drop_duplicates().sort_values(["team", "year"])
    team_year["tournaments_played_before"] = team_year.groupby("team").cumcount()

    df = df.merge(team_year, on=["team", "year"], how="left")
    return df


def compute_champion_features(summary_df: pd.DataFrame) -> pd.DataFrame:
    base = summary_df[["year", "champion", "runner_up", "third_place"]].copy()

    teams = pd.unique(
        pd.concat([base["champion"], base["runner_up"], base["third_place"]])
    )
    teams = pd.Series(teams).dropna().unique()

    rows = []
    for year in base["year"].sort_values().unique():
        yr = base[base["year"] == year].iloc[0]
        for team in teams:
            rows.append(
                {
                    "year": year,
                    "team": team,
                    "is_champion": int(team == yr["champion"]),
                    "is_finalist": int(team in [yr["champion"], yr["runner_up"]]),
                    "is_top3": int(team in [yr["champion"], yr["runner_up"], yr["third_place"]]),
                }
            )

    champ_df = pd.DataFrame(rows)
    champ_df = champ_df.sort_values(["team", "year"]).reset_index(drop=True)

    group = champ_df.groupby("team", group_keys=False)
    champ_df["num_titles_before"] = group["is_champion"].cumsum().shift(1).fillna(0)
    champ_df["num_finals_before"] = group["is_finalist"].cumsum().shift(1).fillna(0)
    champ_df["num_top3_before"] = group["is_top3"].cumsum().shift(1).fillna(0)

    champ_df["has_won_world_cup_before"] = (champ_df["num_titles_before"] > 0).astype(int)

    champ_df["is_defending_champion"] = group["is_champion"].shift(1).fillna(0).astype(int)

    return champ_df[[
        "year",
        "team",
        "has_won_world_cup_before",
        "num_titles_before",
        "num_finals_before",
        "num_top3_before",
        "is_defending_champion",
    ]]


def compute_tournament_context(summary_df: pd.DataFrame) -> pd.DataFrame:
    df = summary_df.copy()

    df["year_normalized"] = (df["year"] - df["year"].min()) / (df["year"].max() - df["year"].min())

    df["tournament_size_category"] = pd.cut(
        df["total_teams"],
        bins=[0, 16, 24, 64],
        labels=["small", "medium", "large"],
        include_lowest=True,
    )

    return df[[
        "year",
        "total_teams",
        "matches_played",
        "goals_scored_tournament",
        "avg_goals_per_game",
        "year_normalized",
        "tournament_size_category",
    ]]


def compute_h2h(long_df: pd.DataFrame) -> pd.DataFrame:
    df = long_df.copy()
    df = df.sort_values(["date", "match_id"]).reset_index(drop=True)

    df["goal_diff"] = df["goals_for"] - df["goals_against"]

    h2h_group = df.groupby(["team", "opponent"], group_keys=False)
    df["h2h_matches_before"] = h2h_group.cumcount()
    df["h2h_wins_before"] = h2h_group["win"].cumsum().shift(1).fillna(0)
    df["h2h_goal_diff_before"] = h2h_group["goal_diff"].cumsum().shift(1).fillna(0)

    df["h2h_win_rate_home_before"] = df["h2h_wins_before"] / df["h2h_matches_before"].replace(0, np.nan)

    df = df.drop(columns=["goal_diff"])

    return df


def compute_elo(long_df: pd.DataFrame, k: float = 20.0, base: float = 1500.0) -> pd.DataFrame:
    df = long_df.copy().sort_values(["date", "match_id"]).reset_index(drop=True)
    ratings = {}
    rows = []

    for row in df.itertuples(index=False):
        team = row.team
        opp = row.opponent

        team_rating = ratings.get(team, base)
        opp_rating = ratings.get(opp, base)

        rows.append({"match_id": row.match_id, "team": team, "elo_before": team_rating})

        if pd.notna(row.goals_for) and pd.notna(row.goals_against):
            if row.goals_for > row.goals_against:
                score = 1.0
            elif row.goals_for < row.goals_against:
                score = 0.0
            else:
                score = 0.5

            expected = 1.0 / (1.0 + 10 ** ((opp_rating - team_rating) / 400))

            ratings[team] = team_rating + k * (score - expected)

    elo_df = pd.DataFrame(rows)
    return elo_df


long_hist = compute_tournament_experience(long_hist)
champion_features = compute_champion_features(summary)
context_features = compute_tournament_context(summary)

long_hist = compute_h2h(long_hist)
elo_df = compute_elo(long_hist)

long_hist = long_hist.merge(elo_df, on=["match_id", "team"], how="left")
long_hist = long_hist.merge(champion_features, on=["year", "team"], how="left")
long_hist = long_hist.merge(context_features, on="year", how="left")

long_hist.head()

,match_id,date,year,stage,tournament_name,host_country,attendance,team,opponent,goals_for,goals_against,is_home,win,draw,loss,points,is_group_stage,is_knockout,total_matches_before,total_wins_before,total_draws_before,total_losses_before,total_goals_for_before,total_goals_against_before,goal_diff_before,total_win_rate_before,goal_diff_per_match_before,goals_per_match_before,conceded_per_match_before,last5_points,last5_goal_diff,last5_win_rate,group_matches_before,group_wins_before,group_stage_win_rate_before,knockout_matches_before,knockout_wins_before,knockout_win_rate_before,times_played_as_home_before,times_played_as_away_before,home_wins_before,away_wins_before,home_win_rate_before,away_win_rate_before,home_away_win_rate_diff,avg_attendance_before,matches_season_before,wins_season_before,goals_scored_season_before,goals_conceded_season_before,win_rate_season_before,avg_goals_scored_season_before,avg_goals_conceded_season_before,goal_diff_season_before,has_won_world_cup_before_x,num_titles_before_x,num_finals_before_x,num_top3_before_x,is_defending_champion_x,tournaments_played_before,h2h_matches_before,h2h_wins_before,h2h_goal_diff_before,h2h_win_rate_home_before,elo_before,has_won_world_cup_before_y,num_titles_before_y,num_finals_before_y,num_top3_before_y,is_defending_champion_y,total_teams,matches_played,goals_scored_tournament,avg_goals_per_game,year_normalized,tournament_size_category
0,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,France,Mexico,4.0,1.0,1,1,0,0,3,True,False,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0,0.0,0.0,0.0,0,0,0,0.0,0.0,NaN,1500.0,1.0,1.0,1.0,1.0,0.0,13,16,70,3.6,0.0,small
1,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,Mexico,France,1.0,4.0,0,0,0,1,0,True,False,0,1.0,0.0,0.0,4.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,-2.0,NaN,1.0,0.0,0.0,0.0,0.0,NaN,1.0,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,0,1.0,4.0,1.0,NaN,NaN,NaN,3.0,1,2.0,4.0,6.0,0,0,0,1.0,3.0,NaN,1500.0,NaN,NaN,NaN,NaN,NaN,13,16,70,3.6,0.0,small
2,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,United States,Belgium,3.0,0.0,1,1,0,0,3,True,False,0,0.0,0.0,1.0,1.0,4.0,-3.0,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,1.0,1.0,1.0,0.0,0.0,NaN,2.0,0.0,1.0,0.0,0.500000,NaN,NaN,NaN,0,0.0,1.0,4.0,NaN,NaN,NaN,-3.0,0,0.0,0.0,0.0,0,0,0,0.0,-3.0,NaN,1500.0,0.0,0.0,0.0,1.0,0.0,13,16,70,3.6,0.0,small
3,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,Belgium,United States,0.0,3.0,0,0,0,1,0,True,False,0,1.0,0.0,0.0,3.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,1.0,1.0,1.0,0.0,0.0,NaN,3.0,0.0,1.0,0.0,0.333333,NaN,NaN,NaN,0,1.0,3.0,0.0,NaN,NaN,NaN,3.0,0,0.0,0.0,1.0,0,0,0,1.0,3.0,NaN,1500.0,0.0,0.0,0.0,1.0,0.0,13,16,70,3.6,0.0,small
4,2,1930-07-14,1930,Group stage,1930 FIFA World Cup,Uruguay,2549.0,Romania,Peru,3.0,1.0,1,1,0,0,3,True,False,0,0.0,0.0,1.0,0.0,3.0,-3.0,NaN,NaN,NaN,NaN,NaN,-2.0,NaN,1.0,1.0,1.0,0.0,0.0,NaN,3.0,1.0,1.0,0.0,0.333333,0.0,0.333333,NaN,0,0.0,0.0,3.0,NaN,NaN,NaN,-3.0,0,0.0,0.0,1.0,0,0,0,0.0,-3.0,NaN,1500.0,NaN,NaN,NaN,NaN,NaN,13,16,70,3.6,0.0,small


In [33]:
# Recompute champion features and Elo without leakage or double-counting.
# Purpose: Use year-level aggregates to build champion history and match-level Elo.

def compute_champion_features(summary_df: pd.DataFrame, teams: pd.Series) -> pd.DataFrame:
    base = summary_df[["year", "champion", "runner_up", "third_place"]].copy()

    long = base.melt(
        id_vars=["year"],
        value_vars=["champion", "runner_up", "third_place"],
        var_name="role",
        value_name="team",
    ).dropna()

    long["is_champion"] = (long["role"] == "champion").astype(int)
    long["is_finalist"] = long["role"].isin(["champion", "runner_up"]).astype(int)
    long["is_top3"] = 1

    teams = pd.Series(teams).dropna().unique()
    years = summary_df["year"].sort_values().unique()

    index = pd.MultiIndex.from_product([teams, years], names=["team", "year"])
    all_team_year = pd.DataFrame(index=index).reset_index()

    merged = all_team_year.merge(long, on=["team", "year"], how="left")
    merged[["is_champion", "is_finalist", "is_top3"]] = merged[[
        "is_champion",
        "is_finalist",
        "is_top3",
    ]].fillna(0)

    group = merged.groupby("team", group_keys=False)
    merged["num_titles_before"] = group["is_champion"].cumsum().shift(1).fillna(0)
    merged["num_finals_before"] = group["is_finalist"].cumsum().shift(1).fillna(0)
    merged["num_top3_before"] = group["is_top3"].cumsum().shift(1).fillna(0)
    merged["has_won_world_cup_before"] = (merged["num_titles_before"] > 0).astype(int)
    merged["is_defending_champion"] = group["is_champion"].shift(1).fillna(0).astype(int)

    return merged[[
        "year",
        "team",
        "has_won_world_cup_before",
        "num_titles_before",
        "num_finals_before",
        "num_top3_before",
        "is_defending_champion",
    ]]


def compute_elo_matchlevel(matches_df: pd.DataFrame, k: float = 20.0, base: float = 1500.0) -> pd.DataFrame:
    df = matches_df.copy().sort_values(["date", "match_id"]).reset_index(drop=True)
    ratings = {}
    rows = []

    for row in df.itertuples(index=False):
        home = row.home_team
        away = row.away_team

        home_rating = ratings.get(home, base)
        away_rating = ratings.get(away, base)

        rows.append({
            "match_id": row.match_id,
            "home_elo_before": home_rating,
            "away_elo_before": away_rating,
        })

        if pd.notna(row.home_goals) and pd.notna(row.away_goals):
            if row.home_goals > row.away_goals:
                score_home = 1.0
            elif row.home_goals < row.away_goals:
                score_home = 0.0
            else:
                score_home = 0.5

            expected_home = 1.0 / (1.0 + 10 ** ((away_rating - home_rating) / 400))
            expected_away = 1.0 - expected_home

            ratings[home] = home_rating + k * (score_home - expected_home)
            ratings[away] = away_rating + k * ((1.0 - score_home) - expected_away)

    return pd.DataFrame(rows)


teams_all = long_hist["team"].unique()
champion_features = compute_champion_features(summary, teams_all)

elo_match = compute_elo_matchlevel(matches)

long_hist = long_hist.drop(columns=[c for c in ["elo_before"] if c in long_hist.columns])

long_hist = long_hist.merge(champion_features, on=["year", "team"], how="left")
long_hist = long_hist.merge(context_features, on="year", how="left")

long_hist = long_hist.merge(
    matches[["match_id", "home_team", "away_team"]], on="match_id", how="left"
)
long_hist = long_hist.merge(elo_match, on="match_id", how="left")
long_hist["elo_before"] = np.where(
    long_hist["team"] == long_hist["home_team"],
    long_hist["home_elo_before"],
    long_hist["away_elo_before"],
)

long_hist = long_hist.drop(columns=["home_elo_before", "away_elo_before"])

long_hist.head()

,match_id,date,year,stage,tournament_name,host_country,attendance,team,opponent,goals_for,goals_against,is_home,win,draw,loss,points,is_group_stage,is_knockout,total_matches_before,total_wins_before,total_draws_before,total_losses_before,total_goals_for_before,total_goals_against_before,goal_diff_before,total_win_rate_before,goal_diff_per_match_before,goals_per_match_before,conceded_per_match_before,last5_points,last5_goal_diff,last5_win_rate,group_matches_before,group_wins_before,group_stage_win_rate_before,knockout_matches_before,knockout_wins_before,knockout_win_rate_before,times_played_as_home_before,times_played_as_away_before,home_wins_before,away_wins_before,home_win_rate_before,away_win_rate_before,home_away_win_rate_diff,avg_attendance_before,matches_season_before,wins_season_before,goals_scored_season_before,goals_conceded_season_before,win_rate_season_before,avg_goals_scored_season_before,avg_goals_conceded_season_before,goal_diff_season_before,has_won_world_cup_before_x,num_titles_before_x,num_finals_before_x,num_top3_before_x,is_defending_champion_x,tournaments_played_before,h2h_matches_before,h2h_wins_before,h2h_goal_diff_before,h2h_win_rate_home_before,has_won_world_cup_before_y,num_titles_before_y,num_finals_before_y,num_top3_before_y,is_defending_champion_y,total_teams_x,matches_played_x,goals_scored_tournament_x,avg_goals_per_game_x,year_normalized_x,tournament_size_category_x,has_won_world_cup_before,num_titles_before,num_finals_before,num_top3_before,is_defending_champion,total_teams_y,matches_played_y,goals_scored_tournament_y,avg_goals_per_game_y,year_normalized_y,tournament_size_category_y,home_team,away_team,elo_before
0,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,France,Mexico,4.0,1.0,1,1,0,0,3,True,False,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0,0.0,0.0,0.0,0,0,0,0.0,0.0,NaN,1.0,1.0,1.0,1.0,0.0,13,16,70,3.6,0.0,small,0,0.0,0.0,0.0,0,13,16,70,3.6,0.0,small,France,Mexico,1500.0
1,0,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,4444.0,Mexico,France,1.0,4.0,0,0,0,1,0,True,False,0,1.0,0.0,0.0,4.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,-2.0,NaN,1.0,0.0,0.0,0.0,0.0,NaN,1.0,0.0,0.0,0.0,0.000000,NaN,NaN,NaN,0,1.0,4.0,1.0,NaN,NaN,NaN,3.0,1,2.0,4.0,6.0,0,0,0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,13,16,70,3.6,0.0,small,1,2.0,4.0,6.0,0,13,16,70,3.6,0.0,small,France,Mexico,1500.0
2,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,United States,Belgium,3.0,0.0,1,1,0,0,3,True,False,0,0.0,0.0,1.0,1.0,4.0,-3.0,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,1.0,1.0,1.0,0.0,0.0,NaN,2.0,0.0,1.0,0.0,0.500000,NaN,NaN,NaN,0,0.0,1.0,4.0,NaN,NaN,NaN,-3.0,0,0.0,0.0,0.0,0,0,0,0.0,-3.0,NaN,0.0,0.0,0.0,1.0,0.0,13,16,70,3.6,0.0,small,0,0.0,0.0,0.0,0,13,16,70,3.6,0.0,small,United States,Belgium,1500.0
3,1,1930-07-13,1930,Group stage,1930 FIFA World Cup,Uruguay,18346.0,Belgium,United States,0.0,3.0,0,0,0,1,0,True,False,0,1.0,0.0,0.0,3.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,1.0,1.0,1.0,0.0,0.0,NaN,3.0,0.0,1.0,0.0,0.333333,NaN,NaN,NaN,0,1.0,3.0,0.0,NaN,NaN,NaN,3.0,0,0.0,0.0,1.0,0,0,0,1.0,3.0,NaN,0.0,0.0,0.0,1.0,0.0,13,16,70,3.6,0.0,small,0,0.0,0.0,1.0,0,13,16,70,3.6,0.0,small,United States,Belgium,1500.0
4,2,1930-07-14,1930,Group stage,1930 FIFA World Cup,Uruguay,2549.0,Romania,Peru,3.0,1.0,1,1,0,0,3,True,False,0,0.0,0.0,1.0,0.0,3.0,-3.0,NaN,NaN,NaN,NaN,NaN,-2.0,NaN,1.0,1.0,1.0,0.0,0.0,NaN,3.0,1.0,1.0,0.0,0.333333,0.0,0.333333,NaN,0,0.0,0.0,3.0,NaN,NaN,NaN,-3.0,0,0.0,0.0,1.0,0,0,0,0.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN,13,16,70,3.6,0.0,small,0,0.0,0.0,1.0,0,13,16,70,3.6,0.0,small,Romania,Peru,1500.0


In [34]:
# Re-assert match_id alignment after feature merges.
matches = matches.reset_index(drop=True).copy()
matches["match_id"] = np.arange(len(matches))
matches

,year,date,tournament_id,tournament_name,match_name,stage,home_team,away_team,home_team_code,away_team_code,home_goals,away_goals,score,result,home_team_win,away_team_win,draw,extra_time,penalty_shootout,score_penalties,stadium,stadium_id,stadium_name,city,host_country,host_team,attendance,match_time,referee,notes,match_id
0,1930,1930-07-13,WC-1930,1930 FIFA World Cup,France v Mexico,Group stage,France,Mexico,FRA,MEX,4.0,1.0,4-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Pocitos,S-193,Estadio Pocitos,Montevideo,Uruguay,0.0,4444.0,15:00,Domingo Lombardi,NaN,0
1,1930,1930-07-13,WC-1930,1930 FIFA World Cup,United States v Belgium,Group stage,United States,Belgium,USA,BEL,3.0,0.0,3-0,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,18346.0,15:00,Jose Macias,NaN,1
2,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Romania v Peru,Group stage,Romania,Peru,ROU,PER,3.0,1.0,3-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Pocitos,S-193,Estadio Pocitos,Montevideo,Uruguay,0.0,2549.0,14:50,Alberto Warnken,NaN,2
3,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Yugoslavia v Brazil,Group stage,Yugoslavia,Brazil,YUG,BRA,2.0,1.0,2-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,24059.0,12:45,Anibal Tejada,NaN,3
4,1930,1930-07-15,WC-1930,1930 FIFA World Cup,Argentina v France,Group stage,Argentina,France,ARG,FRA,1.0,0.0,1-0,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,23409.0,16:00,Gilberto Rego,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,2022,2022-12-10,WC-2022,2022 FIFA World Cup,NaN,quarter-finals,Morocco,Portugal,MAR,PRT,1.0,0.0,1-0,home team win,1.0,0.0,no,no,no,0-0,Al Thumama Stadium,S-112,NaN,Doha,Qatar,NaN,44198.0,18:00,Facundo Tello,NaN,857
858,2022,2022-12-13,WC-2022,2022 FIFA World Cup,NaN,semi-finals,Argentina,Croatia,ARG,HRV,3.0,0.0,3-0,home team win,1.0,0.0,no,no,no,0-0,Lusail Stadium,S-114,NaN,Lusail,Qatar,NaN,88966.0,22:00,Daniele Orsato,NaN,858
859,2022,2022-12-14,WC-2022,2022 FIFA World Cup,NaN,semi-finals,France,Morocco,FRA,MAR,2.0,0.0,2-0,home team win,1.0,0.0,no,no,no,0-0,Al Bayt Stadium,S-107,NaN,Al Khor,Qatar,NaN,68294.0,22:00,César Arturo Ramos,NaN,859
860,2022,2022-12-17,WC-2022,2022 FIFA World Cup,NaN,third-place match,Croatia,Morocco,HRV,MAR,2.0,1.0,2-1,home team win,1.0,0.0,no,no,no,0-0,Khalifa International Stadium,S-110,NaN,Al Rayyan,Qatar,NaN,44137.0,18:00,Abdulrahman Ibrahim Al Jassim,NaN,860


In [35]:
# Add tournament-level context columns to each match record.
matches = matches.merge(context_features, on="year", how="left")
matches

,year,date,tournament_id,tournament_name,match_name,stage,home_team,away_team,home_team_code,away_team_code,home_goals,away_goals,score,result,home_team_win,away_team_win,draw,extra_time,penalty_shootout,score_penalties,stadium,stadium_id,stadium_name,city,host_country,host_team,attendance,match_time,referee,notes,match_id,total_teams,matches_played,goals_scored_tournament,avg_goals_per_game,year_normalized,tournament_size_category
0,1930,1930-07-13,WC-1930,1930 FIFA World Cup,France v Mexico,Group stage,France,Mexico,FRA,MEX,4.0,1.0,4-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Pocitos,S-193,Estadio Pocitos,Montevideo,Uruguay,0.0,4444.0,15:00,Domingo Lombardi,NaN,0,13,16,70,3.6,0.0,small
1,1930,1930-07-13,WC-1930,1930 FIFA World Cup,United States v Belgium,Group stage,United States,Belgium,USA,BEL,3.0,0.0,3-0,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,18346.0,15:00,Jose Macias,NaN,1,13,16,70,3.6,0.0,small
2,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Romania v Peru,Group stage,Romania,Peru,ROU,PER,3.0,1.0,3-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Pocitos,S-193,Estadio Pocitos,Montevideo,Uruguay,0.0,2549.0,14:50,Alberto Warnken,NaN,2,13,16,70,3.6,0.0,small
3,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Yugoslavia v Brazil,Group stage,Yugoslavia,Brazil,YUG,BRA,2.0,1.0,2-1,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,24059.0,12:45,Anibal Tejada,NaN,3,13,16,70,3.6,0.0,small
4,1930,1930-07-15,WC-1930,1930 FIFA World Cup,Argentina v France,Group stage,Argentina,France,ARG,FRA,1.0,0.0,1-0,home team win,1.0,0.0,no,no,no,0-0,Estadio Gran Parque Central,S-192,Estadio Gran Parque Central,Montevideo,Uruguay,0.0,23409.0,16:00,Gilberto Rego,NaN,4,13,16,70,3.6,0.0,small
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,2022,2022-12-10,WC-2022,2022 FIFA World Cup,NaN,quarter-finals,Morocco,Portugal,MAR,PRT,1.0,0.0,1-0,home team win,1.0,0.0,no,no,no,0-0,Al Thumama Stadium,S-112,NaN,Doha,Qatar,NaN,44198.0,18:00,Facundo Tello,NaN,857,32,64,172,2.7,1.0,large
858,2022,2022-12-13,WC-2022,2022 FIFA World Cup,NaN,semi-finals,Argentina,Croatia,ARG,HRV,3.0,0.0,3-0,home team win,1.0,0.0,no,no,no,0-0,Lusail Stadium,S-114,NaN,Lusail,Qatar,NaN,88966.0,22:00,Daniele Orsato,NaN,858,32,64,172,2.7,1.0,large
859,2022,2022-12-14,WC-2022,2022 FIFA World Cup,NaN,semi-finals,France,Morocco,FRA,MAR,2.0,0.0,2-0,home team win,1.0,0.0,no,no,no,0-0,Al Bayt Stadium,S-107,NaN,Al Khor,Qatar,NaN,68294.0,22:00,César Arturo Ramos,NaN,859,32,64,172,2.7,1.0,large
860,2022,2022-12-17,WC-2022,2022 FIFA World Cup,NaN,third-place match,Croatia,Morocco,HRV,MAR,2.0,1.0,2-1,home team win,1.0,0.0,no,no,no,0-0,Khalifa International Stadium,S-110,NaN,Al Rayyan,Qatar,NaN,44137.0,18:00,Abdulrahman Ibrahim Al Jassim,NaN,860,32,64,172,2.7,1.0,large


In [37]:
# Assemble the final ML table with home/away features and target labels.
def assemble_ml_table(matches_df: pd.DataFrame, long_features: pd.DataFrame) -> pd.DataFrame:
    base = matches_df.copy()

    home_features = long_features[long_features["is_home"] == 1].copy()
    away_features = long_features[long_features["is_home"] == 0].copy()

    drop_cols = [
        "opponent",
        "goals_for",
        "goals_against",
        "win",
        "draw",
        "loss",
        "points",
        "is_group_stage",
        "is_knockout",
        "home_team",
        "away_team",
        "date",
        "year",
        "stage",
        "tournament_name",
        "host_country",
        "attendance",
        "is_home",
        "team",
        "tournament_size_category",
        "total_teams",
        "matches_played",
        "goals_scored_tournament",
        "avg_goals_per_game",
        "year_normalized",
    ]

    home_features = home_features.drop(columns=[c for c in drop_cols if c in home_features.columns])
    away_features = away_features.drop(columns=[c for c in drop_cols if c in away_features.columns])

    home_features = home_features.add_prefix("home_")
    away_features = away_features.add_prefix("away_")

    df = base.merge(home_features, left_on="match_id", right_on="home_match_id", how="left")
    df = df.merge(away_features, left_on="match_id", right_on="away_match_id", how="left")

    df = df.drop(columns=["home_match_id", "away_match_id"])

    # Difference features
    df["win_rate_diff"] = df["home_total_win_rate_before"] - df["away_total_win_rate_before"]
    df["goal_diff_diff"] = df["home_goal_diff_before"] - df["away_goal_diff_before"]
    df["form_diff"] = df["home_last5_win_rate"] - df["away_last5_win_rate"]
    df["goals_per_match_diff"] = df["home_goals_per_match_before"] - df["away_goals_per_match_before"]
    df["conceded_per_match_diff"] = df["home_conceded_per_match_before"] - df["away_conceded_per_match_before"]

    df["season_win_rate_diff"] = df["home_win_rate_season_before"] - df["away_win_rate_season_before"]
    df["season_goal_diff_diff"] = df["home_goal_diff_season_before"] - df["away_goal_diff_season_before"]

    df["elo_diff"] = df["home_elo_before"] - df["away_elo_before"]

    # Target (3-class)
    df["result_target"] = np.where(
        df["home_goals"] > df["away_goals"],
        "HomeWin",
        np.where(df["home_goals"] < df["away_goals"], "AwayWin", "Draw"),
    )

    return df


ml_df = assemble_ml_table(matches, long_hist)
ml_df

,year,date,tournament_id,tournament_name,match_name,stage,home_team,away_team,home_team_code,away_team_code,home_goals,away_goals,score,result,home_team_win,away_team_win,draw,extra_time,penalty_shootout,score_penalties,stadium,stadium_id,stadium_name,city,host_country,host_team,attendance,match_time,referee,notes,match_id,total_teams,matches_played,goals_scored_tournament,avg_goals_per_game,year_normalized,tournament_size_category,home_total_matches_before,home_total_wins_before,home_total_draws_before,home_total_losses_before,home_total_goals_for_before,home_total_goals_against_before,home_goal_diff_before,home_total_win_rate_before,home_goal_diff_per_match_before,home_goals_per_match_before,home_conceded_per_match_before,home_last5_points,home_last5_goal_diff,home_last5_win_rate,home_group_matches_before,home_group_wins_before,home_group_stage_win_rate_before,home_knockout_matches_before,home_knockout_wins_before,home_knockout_win_rate_before,home_times_played_as_home_before,home_times_played_as_away_before,home_home_wins_before,home_away_wins_before,home_home_win_rate_before,home_away_win_rate_before,home_home_away_win_rate_diff,home_avg_attendance_before,home_matches_season_before,home_wins_season_before,home_goals_scored_season_before,home_goals_conceded_season_before,home_win_rate_season_before,home_avg_goals_scored_season_before,home_avg_goals_conceded_season_before,home_goal_diff_season_before,home_has_won_world_cup_before_x,home_num_titles_before_x,home_num_finals_before_x,home_num_top3_before_x,home_is_defending_champion_x,home_tournaments_played_before,home_h2h_matches_before,home_h2h_wins_before,home_h2h_goal_diff_before,home_h2h_win_rate_home_before,home_has_won_world_cup_before_y,home_num_titles_before_y,home_num_finals_before_y,home_num_top3_before_y,home_is_defending_champion_y,home_total_teams_x,home_matches_played_x,home_goals_scored_tournament_x,home_avg_goals_per_game_x,home_year_normalized_x,home_tournament_size_category_x,home_has_won_world_cup_before,home_num_titles_before,home_num_finals_before,home_num_top3_before,home_is_defending_champion,home_total_teams_y,home_matches_played_y,home_goals_scored_tournament_y,home_avg_goals_per_game_y,home_year_normalized_y,home_tournament_size_category_y,home_elo_before,away_total_matches_before,away_total_wins_before,away_total_draws_before,away_total_losses_before,away_total_goals_for_before,away_total_goals_against_before,away_goal_diff_before,away_total_win_rate_before,away_goal_diff_per_match_before,away_goals_per_match_before,away_conceded_per_match_before,away_last5_points,away_last5_goal_diff,away_last5_win_rate,away_group_matches_before,away_group_wins_before,away_group_stage_win_rate_before,away_knockout_matches_before,away_knockout_wins_before,away_knockout_win_rate_before,away_times_played_as_home_before,away_times_played_as_away_before,away_home_wins_before,away_away_wins_before,away_home_win_rate_before,away_away_win_rate_before,away_home_away_win_rate_diff,away_avg_attendance_before,away_matches_season_before,away_wins_season_before,away_goals_scored_season_before,away_goals_conceded_season_before,away_win_rate_season_before,away_avg_goals_scored_season_before,away_avg_goals_conceded_season_before,away_goal_diff_season_before,away_has_won_world_cup_before_x,away_num_titles_before_x,away_num_finals_before_x,away_num_top3_before_x,away_is_defending_champion_x,away_tournaments_played_before,away_h2h_matches_before,away_h2h_wins_before,away_h2h_goal_diff_before,away_h2h_win_rate_home_before,away_has_won_world_cup_before_y,away_num_titles_before_y,away_num_finals_before_y,away_num_top3_before_y,away_is_defending_champion_y,away_total_teams_x,away_matches_played_x,away_goals_scored_tournament_x,away_avg_goals_per_game_x,away_year_normalized_x,away_tournament_size_category_x,away_has_won_world_cup_before,away_num_titles_before,away_num_finals_before,away_num_top3_before,away_is_defending_champion,away_total_teams_y,away_matches_played_y,away_goals_scored_to

# No-Leakage Validation

Purpose: Explicitly exclude post-match fields and confirm the feature list is safe.

In [38]:
# Define allowed features and remove all post-match leakage columns.
LEAKAGE_COLS = {
    "home_goals",
    "away_goals",
    "score",
    "result",
    "home_team_win",
    "away_team_win",
    "draw",
    "extra_time",
    "penalty_shootout",
    "score_penalties",
}

# Base categorical features allowed
CATEGORICAL_COLS = [
    "stage",
    "tournament_name",
    "host_country",
    "home_team",
    "away_team",
    "tournament_size_category",
]

# Numeric engineered features (derived in long_hist with home_/away_ prefixes)
ENGINEERED_PREFIXES = [
    "home_",
    "away_",
]

feature_cols = [
    c
    for c in ml_df.columns
    if (
        c in CATEGORICAL_COLS
        or any(c.startswith(p) for p in ENGINEERED_PREFIXES)
        or c in [
            "win_rate_diff",
            "goal_diff_diff",
            "form_diff",
            "goals_per_match_diff",
            "conceded_per_match_diff",
            "season_win_rate_diff",
            "season_goal_diff_diff",
            "elo_diff",
            "total_teams",
            "matches_played",
            "goals_scored_tournament",
            "avg_goals_per_game",
            "year_normalized",
        ]
    )
]

# Remove leakage columns and any accidental target
feature_cols = [c for c in feature_cols if c not in LEAKAGE_COLS and c != "result_target"]

leakage_in_features = set(feature_cols) & LEAKAGE_COLS
print("Leakage columns in features:", leakage_in_features)
print("Feature count:", len(feature_cols))

Leakage columns in features: set()
Feature count: 159


In [39]:
# Export ML table to CSV
output_path = ROOT / "data" / "ml_df.csv"
ml_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: /Users/jeanphilippeauguste/Downloads/DS4-World-Cup-Project-Phase/data/ml_df.csv
